In [ ]:
import pandas as pd

# =============================================
# 데이터 불러오기
# =============================================

# orders.csv 파일 불러오기
df = pd.read_csv('../data/orders.csv')

# 잘 불러왔는지 확인
print('데이터 불러오기 완료!')
print('데이터 크기:', df.shape)
print(df.head())

# 결측값 확인
print('결측값 확인:')
print(df.isnull().sum())


# =============================================
# 메모리 최적화
# (데이터가 너무 크기 때문에 꼭 필요!)
# =============================================

# 최적화 전에 메모리 얼마나 쓰는지 먼저 확인
before = df.memory_usage(deep=True).sum() / 1024**2
print('최적화 전 메모리:', round(before, 1), 'MB')

# 숫자 범위가 작은 컬럼은 더 작은 타입으로 바꿔주기
df['order_dow']              = df['order_dow'].astype('int8')
df['order_hour_of_day']      = df['order_hour_of_day'].astype('int8')
df['order_number']           = df['order_number'].astype('int16')
df['days_since_prior_order'] = df['days_since_prior_order'].astype('float32')

# 최적화 후 메모리 얼마나 줄었는지 확인
after = df.memory_usage(deep=True).sum() / 1024**2
print('최적화 후 메모리:', round(after, 1), 'MB')
print('줄어든 양:', round(before - after, 1), 'MB')


# =============================================
# Feature 1 : 유저별 총 주문 건수
# =============================================

총주문건수 = df.groupby('user_id')['order_id'].count().reset_index()
총주문건수.columns = ['user_id', 'total_orders']

print('유저별 총 주문 건수:')
print(총주문건수.head())
print('전체 유저 수:', len(총주문건수))


# =============================================
# Feature 2 : 유저별 평균 재구매 주기
# =============================================

재구매데이터 = df[df['days_since_prior_order'].notnull()]
재구매빈도 = 재구매데이터.groupby('user_id')['days_since_prior_order'].mean().reset_index()
재구매빈도.columns = ['user_id', 'avg_days_between_orders']

print('유저별 평균 재구매 주기:')
print(재구매빈도.head())


# =============================================
# Feature 3 : 유저별 선호 요일
# =============================================

가장많은요일 = df.groupby('user_id')['order_dow'].agg(lambda x: x.mode()[0]).reset_index()
가장많은요일.columns = ['user_id', 'favorite_dow']

print('유저별 선호 요일:')
print(가장많은요일.head())


# =============================================
# Feature 4 : 유저별 선호 시간대
# =============================================

가장많은시간 = df.groupby('user_id')['order_hour_of_day'].agg(lambda x: x.mode()[0]).reset_index()
가장많은시간.columns = ['user_id', 'favorite_hour']

print('유저별 선호 시간대:')
print(가장많은시간.head())


# =============================================
# Feature 5 : 재구매 주기 규칙성
# =============================================

주기규칙성 = 재구매데이터.groupby('user_id')['days_since_prior_order'].std().reset_index()
주기규칙성.columns = ['user_id', 'std_days_between_orders']

print('유저별 재구매 주기 규칙성:')
print(주기규칙성.head())


# =============================================
# Feature 6 : 충성도 지표
# =============================================

최근주문번호 = df.groupby('user_id')['order_number'].max().reset_index()
최근주문번호.columns = ['user_id', 'max_order_number']

print('유저별 충성도 지표:')
print(최근주문번호.head())


# =============================================
# 전부 합치기
# =============================================

user_features = 총주문건수.merge(재구매빈도,   on='user_id')
user_features = user_features.merge(가장많은요일, on='user_id')
user_features = user_features.merge(가장많은시간, on='user_id')
user_features = user_features.merge(주기규칙성,   on='user_id')
user_features = user_features.merge(최근주문번호, on='user_id')

print('합친 후 user_features:')
print(user_features.head(10))
print('크기:', user_features.shape)


# =============================================
# 합친 후 메모리 최적화
# =============================================

user_features['total_orders']            = user_features['total_orders'].astype('int16')
user_features['avg_days_between_orders'] = user_features['avg_days_between_orders'].astype('float32')
user_features['favorite_dow']            = user_features['favorite_dow'].astype('int8')
user_features['favorite_hour']           = user_features['favorite_hour'].astype('int8')
user_features['std_days_between_orders'] = user_features['std_days_between_orders'].astype('float32')
user_features['max_order_number']        = user_features['max_order_number'].astype('int16')

print('최종 메모리:', round(user_features.memory_usage(deep=True).sum() / 1024**2, 1), 'MB')


# =============================================
# CSV 저장
# =============================================

user_features.to_csv('../data/user_features.csv', index=False)
print('저장 완료! → data/user_features.csv')